<a href="https://colab.research.google.com/github/kssoftage/retropad-build123d/blob/main/RetroPad_build123d_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RetroPad — build123d Pipeline

**Goal:** STL → zero-diff STEP/STL via OCP sew+solid → build123d

| Stage | Status |
|-------|--------|
| Individual parts (4x) | ✅ Zero diff verified |
| Assembly positioning | ⚠️ TODO: button positions from RetroPad.step |

**Source:** https://github.com/jtgans/RetroPad

## Cell 1 — Install & Imports

In [ ]:
# Install
!pip install cadquery trimesh shapely -q

import trimesh
import numpy as np
from shapely.geometry import Polygon

from OCP.StlAPI import StlAPI_Reader
from OCP.TopoDS import TopoDS_Shape, TopoDS
from OCP.BRepBuilderAPI import BRepBuilderAPI_Sewing, BRepBuilderAPI_MakeSolid
from OCP.ShapeFix import ShapeFix_Solid
from OCP.TopAbs import TopAbs_SHELL
from OCP.TopExp import TopExp_Explorer
from OCP.GProp import GProp_GProps
from OCP.BRepGProp import brepgprop_VolumeProperties
from OCP.STEPControl import STEPControl_Writer
from OCP.IFSelect import IFSelect_RetDone

from build123d import import_step, export_step, export_stl, Compound, Location, Vector

STL_DIR = '/content/RetroPad/case'
print('✅ Imports OK')

## Cell 2 — Clone Repo

In [ ]:
import os
if not os.path.exists('/content/RetroPad'):
    !git clone https://github.com/jtgans/RetroPad /content/RetroPad
else:
    print('Already cloned')

!ls /content/RetroPad/case/

## Cell 3 — STL Geometry Analysis

Measures reference volumes and bounding boxes for all 4 parts.
These values are used for zero-diff verification in Cell 5.

In [ ]:
PARTS = {
    'bottom': 'RetroPad - Bottom Shell.stl',
    'top':    'RetroPad - Top Shell.stl',
    'button': 'RetroPad - Button.stl',
    'dpad':   'RetroPad - D-Pad.stl',
}

ref = {}
print(f'{'Part':<10} {'Volume (mm³)':>14}  {'BBox (mm)'}')
print('-' * 55)
for name, fname in PARTS.items():
    mesh = trimesh.load(f'{STL_DIR}/{fname}')
    v = mesh.vertices
    dims = v.max(axis=0) - v.min(axis=0)
    cx = float((v[:,0].min() + v[:,0].max()) / 2)
    cy = float((v[:,1].min() + v[:,1].max()) / 2)
    ref[name] = {
        'volume': float(mesh.volume),
        'bbox':   dims.tolist(),
        'centre': (cx, cy),
        'zrange': (float(v[:,2].min()), float(v[:,2].max())),
    }
    print(f'{name:<10} {mesh.volume:>14.2f}  {dims[0]:.1f} × {dims[1]:.1f} × {dims[2]:.2f}')

print(f"\nButton STL centre in assembly frame: {ref['button']['centre']}")
print(f"D-Pad  STL centre in assembly frame: {ref['dpad']['centre']}")

## Cell 4 — OCP Sew+Solid Pipeline

**Why not raw STL import?**
`StlAPI_Reader` returns a triangulated shell — build123d sees `volume=0, solids=0`.
Fix: sew triangles → closed shell → cast to solid → ShapeFix.

**Result:** All 4 parts export as valid STEP solids with exact STL volume.

In [ ]:
def stl_to_solid_step(stl_path, step_path, tol=0.01):
    """STL → OCP sew+solid → STEP. Returns solid volume (mm³)."""
    # 1. Read STL triangles
    raw = TopoDS_Shape()
    StlAPI_Reader().Read(raw, stl_path)

    # 2. Sew triangles into closed shell
    sewer = BRepBuilderAPI_Sewing(tol)
    sewer.Add(raw)
    sewer.Perform()
    sewn = sewer.SewedShape()

    # 3. Cast TopoDS_Shape → TopoDS_Shell (critical step)
    explorer = TopExp_Explorer(sewn, TopAbs_SHELL)
    shell = TopoDS.Shell_s(explorer.Current())

    # 4. Shell → Solid
    maker = BRepBuilderAPI_MakeSolid(shell)
    maker.Build()
    solid = maker.Solid()

    # 5. Fix orientation (reversed shell gives negative volume)
    fixer = ShapeFix_Solid(solid)
    fixer.Perform()
    fixed = fixer.Solid()

    # 6. Volume check
    props = GProp_GProps()
    brepgprop_VolumeProperties(fixed, props)
    vol = abs(props.Mass())

    # 7. Export STEP
    writer = STEPControl_Writer()
    writer.Transfer(fixed, 0)
    assert writer.Write(step_path) == IFSelect_RetDone

    return vol


print(f'{'Part':<10} {'Ref Vol':>12}  {'Solid Vol':>12}  {'Diff':>10}  Status')
print('-' * 60)
for name, fname in PARTS.items():
    stl_path  = f'{STL_DIR}/{fname}'
    step_path = f'/content/{name}_solid.step'
    vol = stl_to_solid_step(stl_path, step_path)
    r   = ref[name]['volume']
    diff = vol - r
    ok  = '✅' if abs(diff) < 0.01 else '❌'
    print(f'{name:<10} {r:>12.2f}  {vol:>12.2f}  {diff:>+10.2f}  {ok}')

## Cell 5 — build123d Import + Export Verify

Import each STEP into build123d, confirm `solids=1` and volume matches reference.
Export back to STEP and STL — final zero-diff check.

In [ ]:
b123d_parts = {}

print(f'{'Part':<10} {'Solids':>7}  {'b123d Vol':>12}  {'Diff':>10}  Status')
print('-' * 58)
for name in PARTS:
    s = import_step(f'/content/{name}_solid.step')
    n_solids = len(s.solids())
    vol  = s.volume
    r    = ref[name]['volume']
    diff = vol - r
    ok   = '✅' if abs(diff) < 0.01 else '❌'
    print(f'{name:<10} {n_solids:>7}  {vol:>12.2f}  {diff:>+10.2f}  {ok}')
    b123d_parts[name] = s
    export_step(s, f'/content/{name}_b123d_final.step')
    export_stl(s,  f'/content/{name}_b123d_final.stl')

print('\nAll individual parts exported.')

## Cell 6 — Assembly

### Known positions (from STL vertex analysis)
| Part | Centre X | Centre Y |
|------|----------|----------|
| Button (single STL) | 38.000 | 8.511 |
| D-Pad | -38.000 | 0.001 |

### ⚠️ TODO — Button grid positions
Need to extract 4 button hole centres from `/content/RetroPad/RetroPad.step`  
(full assembly STEP available — use OCP `TopExp_Explorer` → volume filter ~1150 mm³ → CentreOfMass per solid)

Current cell uses **placeholder** positions until that is resolved.

In [ ]:
# ── TODO: replace with positions extracted from RetroPad.step ──────────────
# from OCP.STEPControl import STEPControl_Reader
# from OCP.TopAbs import TopAbs_SOLID
# reader = STEPControl_Reader()
# reader.ReadFile('/content/RetroPad/RetroPad.step')
# reader.TransferRoots()
# shape = reader.OneShape()
# ... filter solids by volume ~1150 → get 4 CentreOfMass XY
# ───────────────────────────────────────────────────────────────────────────

BUTTON_BASE_CX, BUTTON_BASE_CY = ref['button']['centre']

# PLACEHOLDER — positions not verified
BUTTON_POSITIONS = [
    ( 38.0,   8.511),   # A
    ( 49.0,   0.0  ),   # B  ← unverified
    ( 38.0,  -8.511),   # X  ← unverified
    ( 27.0,   0.0  ),   # Y  ← unverified
]

btn    = b123d_parts['button']
bottom = b123d_parts['bottom']
top    = b123d_parts['top']
dpad   = b123d_parts['dpad']

buttons = [
    btn.moved(Location(Vector(cx - BUTTON_BASE_CX, cy - BUTTON_BASE_CY, 0.0)))
    for cx, cy in BUTTON_POSITIONS
]

assembly = Compound([bottom, top, dpad, *buttons])
export_step(assembly, '/content/retropad_assembly_final.step')
export_stl(assembly,  '/content/retropad_assembly_final.stl')

# Volume check
ref_total = ref['bottom']['volume'] + ref['top']['volume'] + \
            ref['dpad']['volume']   + 4 * ref['button']['volume']
asm_vol   = trimesh.load('/content/retropad_assembly_final.stl').volume
diff      = asm_vol - ref_total

print(f'Ref total  : {ref_total:.2f} mm³')
print(f'Assembly   : {asm_vol:.2f} mm³')
print(f'Diff       : {diff:+.2f} mm³')
print(f'Status     : {"✅ ZERO DIFF" if abs(diff) < 0.1 else "⚠️ CHECK (likely star artifacts from overlapping faces)"}')